# Side Effect Unique263 Training

Baseline unique-263 LightGBM training. Run one seed (42, 43 or 44) per session. Requires prepared X/Y matrices and the corresponding label mapping. This is the baseline protocol, separate from subsequent hybrid tuning.


## Stage 1

Source: `side_effect_and_baseline_history`, cell 16 (zero-based).


In [ ]:
# ============================================================
# FINAL UNIQUE-263 LIGHTGBM TRAINING
# Run ONE seed per notebook
#
# Notebook 1: SEED = 42
# Notebook 2: SEED = 43
# Notebook 3: SEED = 44
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

!pip -q install lightgbm==4.6.0

from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import lightgbm as lgb

import json
import hashlib
import time
import gc
import re
import platform


# ============================================================
# CONFIG
# ============================================================

SEED = 42   # <-- CHANGE ONLY THIS: 42 / 43 / 44

N_LABELS = 263
EARLY_STOPPING_ROUNDS = 50


FEATURE_DIR = Path(
    "/content/drive/MyDrive/"
    "DDI_SIDE_EFFECT_84_CORRECTED/"
    "morgan_features_84_263_UNIQUE"
)


OUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "DDI_SIDE_EFFECT_84_CORRECTED/"
    "FINAL_UNIQUE263_LGBM"
)


SEED_DIR = (
    OUT_ROOT /
    f"seed_{SEED}"
)

MODEL_DIR = (
    SEED_DIR /
    "models"
)


MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 100)
print("FINAL UNIQUE-263 TRAINING")
print("=" * 100)

print("SEED:", SEED)
print("FEATURE_DIR:", FEATURE_DIR)
print("SEED_DIR:", SEED_DIR)


# ============================================================
# LOAD FEATURES
# ============================================================

X_train = (
    sp.load_npz(
        FEATURE_DIR / "X_train.npz"
    )
    .tocsr()
    .astype(np.float32)
)

X_cal = (
    sp.load_npz(
        FEATURE_DIR / "X_calibration.npz"
    )
    .tocsr()
    .astype(np.float32)
)

X_test = (
    sp.load_npz(
        FEATURE_DIR / "X_test.npz"
    )
    .tocsr()
    .astype(np.float32)
)


Y_train = np.load(
    FEATURE_DIR / "Y_train.npy",
    mmap_mode="r"
)

Y_cal = np.load(
    FEATURE_DIR / "Y_calibration.npy",
    mmap_mode="r"
)

Y_test = np.load(
    FEATURE_DIR / "Y_test.npy",
    mmap_mode="r"
)


label_map = (
    pd.read_csv(
        FEATURE_DIR / "label_mapping.csv"
    )
    .sort_values("label_id")
    .reset_index(drop=True)
)

LABELS = (
    label_map["label"]
    .astype(str)
    .tolist()
)


print("\nShapes:")

print(
    "X_train:",
    X_train.shape,
    X_train.dtype
)

print(
    "X_cal:",
    X_cal.shape,
    X_cal.dtype
)

print(
    "X_test:",
    X_test.shape,
    X_test.dtype
)

print(
    "Y_train:",
    Y_train.shape
)

print(
    "Y_cal:",
    Y_cal.shape
)

print(
    "Y_test:",
    Y_test.shape
)

print(
    "Labels:",
    len(LABELS)
)


assert X_train.shape == (134249, 4096)
assert X_cal.shape == (28768, 4096)
assert X_test.shape == (28768, 4096)

assert Y_train.shape == (134249, 263)
assert Y_cal.shape == (28768, 263)
assert Y_test.shape == (28768, 263)

assert len(LABELS) == 263


# ============================================================
# CONFIRM 3 NEW LABELS
# ============================================================

for target in [
    "anaphylaxis",
    "hyperglycemia",
    "tinnitus"
]:

    ids = label_map.loc[
        label_map["label"] == target,
        "label_id"
    ].tolist()

    print(
        target,
        "->",
        ids
    )

    assert len(ids) == 1


# ============================================================
# LIGHTGBM SETTINGS
# ============================================================

BASE_PARAMS = {

    "n_estimators": 500,

    "learning_rate": 0.05,

    "num_leaves": 63,

    "min_child_samples": 30,

    "subsample": 0.85,

    "subsample_freq": 1,

    "colsample_bytree": 0.85,

    "reg_lambda": 1.0,

    "objective": "binary",

    "n_jobs": 2,

    "verbosity": -1,
}


# ============================================================
# HASH
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


# ============================================================
# TRAINING CONFIG
# ============================================================

config = {

    "seed": SEED,

    "final_unique_263":
        True,

    "label_count":
        263,

    "interaction_classes":
        84,

    "feature_dimension":
        4096,

    "lightgbm_version":
        lgb.__version__,

    "python_version":
        platform.python_version(),

    "parameters":
        BASE_PARAMS,

    "early_stopping_rounds":
        EARLY_STOPPING_ROUNDS,

    "class_weight_formula":
        (
            "sqrt((train_negatives + 1) / "
            "(train_positives + 1))"
        ),

    "label_seed_rule":
        "seed + label_id",

    "calibration_used_for_early_stopping":
        True,

    "test_used_for_training":
        False,

    "replacement_labels": [
        "anaphylaxis",
        "hyperglycemia",
        "tinnitus"
    ]
}


with open(
    SEED_DIR /
    "training_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# PROBABILITY MATRICES
# ============================================================

CAL_PATH = (
    SEED_DIR /
    "calibration_probabilities.npy"
)

TEST_PATH = (
    SEED_DIR /
    "test_probabilities.npy"
)


def open_probabilities(
    path,
    rows
):

    if path.exists():

        arr = np.lib.format.open_memmap(
            path,
            mode="r+"
        )

        if arr.shape != (
            rows,
            263
        ):

            raise ValueError(
                f"Wrong shape: "
                f"{path} -> {arr.shape}"
            )

        return arr


    arr = np.lib.format.open_memmap(

        path,

        mode="w+",

        dtype=np.float32,

        shape=(
            rows,
            263
        )
    )

    arr[:] = np.nan

    arr.flush()

    return arr


cal_probs = open_probabilities(
    CAL_PATH,
    X_cal.shape[0]
)

test_probs = open_probabilities(
    TEST_PATH,
    X_test.shape[0]
)


# ============================================================
# RESUME STATUS
# ============================================================

pattern = re.compile(
    r"^label_(\d{3})\.txt$"
)

existing = []

for p in MODEL_DIR.iterdir():

    m = pattern.match(
        p.name
    )

    if m:

        existing.append(
            int(
                m.group(1)
            )
        )


print("\nExisting models:", len(existing), "/263")


# ============================================================
# TRAIN LOOP
# ============================================================

run_start = time.time()


for label_id in range(263):

    label = LABELS[label_id]


    model_path = (
        MODEL_DIR /
        f"label_{label_id:03d}.txt"
    )

    meta_path = (
        MODEL_DIR /
        f"label_{label_id:03d}.json"
    )


    print("\n" + "=" * 100)

    print(
        f"SEED {SEED} | "
        f"Label {label_id + 1}/263 "
        f"(ID={label_id:03d})"
    )

    print(
        "Label:",
        repr(label)
    )


    # ========================================================
    # RESUME EXISTING MODEL
    # ========================================================

    if model_path.exists():

        print(
            " Existing model — skip training"
        )

        booster = lgb.Booster(
            model_file=str(
                model_path
            )
        )

        iteration = int(
            booster.current_iteration()
        )

        cal_probs[
            :,
            label_id
        ] = booster.predict(
            X_cal,
            num_iteration=iteration
        ).astype(
            np.float32
        )

        test_probs[
            :,
            label_id
        ] = booster.predict(
            X_test,
            num_iteration=iteration
        ).astype(
            np.float32
        )

        cal_probs.flush()
        test_probs.flush()

        del booster
        gc.collect()

        continue


    # ========================================================
    # TARGET
    # ========================================================

    y_train = np.asarray(
        Y_train[:, label_id],
        dtype=np.int32
    )

    y_cal = np.asarray(
        Y_cal[:, label_id],
        dtype=np.int32
    )

    y_test = np.asarray(
        Y_test[:, label_id],
        dtype=np.int32
    )


    train_pos = int(
        y_train.sum()
    )

    train_neg = int(
        len(y_train) -
        train_pos
    )

    cal_pos = int(
        y_cal.sum()
    )

    test_pos = int(
        y_test.sum()
    )


    if train_pos == 0:

        raise ValueError(
            f"Label {label_id} "
            "has zero train positives"
        )


    scale_pos_weight = float(

        np.sqrt(

            (train_neg + 1)
            /
            (train_pos + 1)

        )
    )


    print(
        "Train +:",
        train_pos,
        "| Cal +:",
        cal_pos,
        "| Test +:",
        test_pos
    )

    print(
        "scale_pos_weight:",
        round(
            scale_pos_weight,
            6
        )
    )


    # ========================================================
    # MODEL
    # ========================================================

    label_seed = (
        SEED +
        label_id
    )


    params = dict(
        BASE_PARAMS
    )

    params.update({

        "random_state":
            label_seed,

        "scale_pos_weight":
            scale_pos_weight,
    })


    model = lgb.LGBMClassifier(
        **params
    )


    start = time.time()


    model.fit(

        X_train,

        y_train,

        eval_set=[
            (
                X_cal,
                y_cal
            )
        ],

        eval_metric=
            "binary_logloss",

        callbacks=[

            lgb.early_stopping(

                stopping_rounds=
                    EARLY_STOPPING_ROUNDS,

                verbose=False

            )

        ]
    )


    seconds = (
        time.time() -
        start
    )


    booster = model.booster_


    best_iteration = int(

        booster.best_iteration

        if booster.best_iteration

        else

        booster.current_iteration()
    )


    # ========================================================
    # SAVE MODEL
    # ========================================================

    booster.save_model(

        str(
            model_path
        ),

        num_iteration=
            best_iteration
    )


    assert model_path.exists()


    # ========================================================
    # PREDICTIONS
    # ========================================================

    cal_pred_prob = booster.predict(
        X_cal,
        num_iteration=
            best_iteration
    )


    test_pred_prob = booster.predict(
        X_test,
        num_iteration=
            best_iteration
    )


    cal_probs[
        :,
        label_id
    ] = cal_pred_prob.astype(
        np.float32
    )


    test_probs[
        :,
        label_id
    ] = test_pred_prob.astype(
        np.float32
    )


    cal_probs.flush()
    test_probs.flush()


    # ========================================================
    # METADATA
    # ========================================================

    metadata = {

        "label_id":
            label_id,

        "label":
            label,

        "seed":
            SEED,

        "label_seed":
            label_seed,

        "train_positive_rows":
            train_pos,

        "train_negative_rows":
            train_neg,

        "calibration_positive_rows":
            cal_pos,

        "test_positive_rows":
            test_pos,

        "scale_pos_weight":
            scale_pos_weight,

        "saved_iterations":
            best_iteration,

        "training_seconds":
            float(
                seconds
            ),

        "model_sha256":
            sha256_file(
                model_path
            )
    }


    with open(
        meta_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False
        )


    print(
        " Saved:",
        model_path.name
    )

    print(
        "Best iteration:",
        best_iteration
    )

    print(
        "Seconds:",
        round(
            seconds,
            2
        )
    )


    # ========================================================
    # CLEANUP
    # ========================================================

    del model
    del booster

    del y_train
    del y_cal
    del y_test

    del cal_pred_prob
    del test_pred_prob

    gc.collect()


# ============================================================
# FINAL FLUSH
# ============================================================

cal_probs.flush()
test_probs.flush()


# ============================================================
# FINAL AUDIT
# ============================================================

txt_pattern = re.compile(
    r"^label_(\d{3})\.txt$"
)

json_pattern = re.compile(
    r"^label_(\d{3})\.json$"
)


txt_ids = set()
json_ids = set()
noncanonical = []


for p in MODEL_DIR.iterdir():

    mt = txt_pattern.match(
        p.name
    )

    mj = json_pattern.match(
        p.name
    )

    if mt:

        txt_ids.add(
            int(
                mt.group(1)
            )
        )

    elif mj:

        json_ids.add(
            int(
                mj.group(1)
            )
        )

    elif p.name.startswith(
        "label_"
    ):

        noncanonical.append(
            p.name
        )


expected = set(
    range(263)
)


missing_txt = sorted(
    expected -
    txt_ids
)

missing_json = sorted(
    expected -
    json_ids
)


cal_check = np.load(
    CAL_PATH,
    mmap_mode="r"
)

test_check = np.load(
    TEST_PATH,
    mmap_mode="r"
)


cal_nan = int(
    np.isnan(
        cal_check
    ).sum()
)

test_nan = int(
    np.isnan(
        test_check
    ).sum()
)


print("\n" + "=" * 110)
print(
    f"FINAL SEED {SEED} AUDIT"
)
print("=" * 110)

print(
    "Models:",
    len(txt_ids)
)

print(
    "Metadata:",
    len(json_ids)
)

print(
    "Missing models:",
    missing_txt
)

print(
    "Missing metadata:",
    missing_json
)

print(
    "Non-canonical:",
    noncanonical
)

print(
    "Calibration shape:",
    cal_check.shape
)

print(
    "Test shape:",
    test_check.shape
)

print(
    "Calibration NaNs:",
    cal_nan
)

print(
    "Test NaNs:",
    test_nan
)


ready = (

    len(txt_ids) == 263

    and

    len(json_ids) == 263

    and

    not missing_txt

    and

    not missing_json

    and

    not noncanonical

    and

    cal_check.shape ==
    (28768, 263)

    and

    test_check.shape ==
    (28768, 263)

    and

    cal_nan == 0

    and

    test_nan == 0
)


if ready:

    completion = {

        "seed":
            SEED,

        "completed":
            True,

        "final_unique_263":
            True,

        "model_count":
            263,

        "metadata_count":
            263,

        "calibration_probability_shape":
            [28768, 263],

        "test_probability_shape":
            [28768, 263],

        "completed_at":
            time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }


    with open(
        SEED_DIR /
        "training_complete.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            completion,
            f,
            indent=2
        )


    print(
        f"\n FINAL UNIQUE263 SEED "
        f"{SEED} COMPLETE — 263/263"
    )

else:

    print(
        f"\n Seed {SEED} incomplete"
    )


print(
    "\nSaved in:"
)

print(
    SEED_DIR
)